# TPC_RP — Modified Algorithm

This notebook builds on `tpcrp_original.ipynb` and explores **modifications** to the
TPC_RP algorithm. Run the original notebook first to generate:
- `results/simclr_encoder.pt`
- `results/train_features.npy` / `results/test_features.npy`
- `results/tpcrp_original_history.json`

## Modifications explored
1. **No random projection** — cluster in full feature space (ablation)
2. **Varied projection dimension** — compare proj_dim ∈ {32, 64, 128}
3. **Random baseline** — uniform random sampling for comparison

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.utils import set_seed, load_results, save_results, plot_comparison, log
from src.active_learning import TypiClustSelector, run_active_learning_loop
from src.classifier import train_linear_probe, evaluate_linear_probe

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)
print(f'Device: {DEVICE}')

## 1. Load Pre-computed Features

In [ ]:
train_feats  = np.load('../results/train_features.npy')
train_labels = np.load('../results/train_labels.npy')
test_feats   = np.load('../results/test_features.npy')
test_labels  = np.load('../results/test_labels.npy')

print(f'Train: {train_feats.shape}  |  Test: {test_feats.shape}')

## 2. Shared Helpers

In [ ]:
INITIAL_BUDGET = 10
QUERY_BUDGET   = 10
N_ROUNDS       = 20
PROBE_EPOCHS   = 100
PROBE_LR       = 1e-3

def train_fn(labelled_idx, labels):
    return train_linear_probe(
        train_features = train_feats[labelled_idx],
        train_labels   = labels,
        epochs         = PROBE_EPOCHS,
        lr             = PROBE_LR,
        device         = DEVICE,
    )

def eval_fn(head):
    return evaluate_linear_probe(head, test_feats, test_labels, device=DEVICE)

def run(selector, name):
    log(f'Running {name}...')
    hist = run_active_learning_loop(
        features       = train_feats,
        labels         = train_labels,
        selector       = selector,
        initial_budget = INITIAL_BUDGET,
        query_budget   = QUERY_BUDGET,
        n_rounds       = N_ROUNDS,
        train_fn       = train_fn,
        eval_fn        = eval_fn,
        seed           = SEED,
    )
    save_results(hist, f'../results/{name.replace(" ", "_")}_history.json')
    return hist

## 3. Modification 1 — No Random Projection (Ablation)

In [ ]:
# proj_dim=None disables random projection; clustering is done in the full 512-d space.
sel_no_proj = TypiClustSelector(proj_dim=None, knn_k=20, seed=SEED)
hist_no_proj = run(sel_no_proj, 'TPC_no_proj')

## 4. Modification 2 — Varied Projection Dimension

In [ ]:
histories_proj = {}
for d in [32, 64, 128]:
    sel = TypiClustSelector(proj_dim=d, knn_k=20, seed=SEED)
    histories_proj[f'TPC_RP_proj{d}'] = run(sel, f'TPC_RP_proj{d}')

## 5. Random Baseline

In [ ]:
import numpy as np

class RandomSelector:
    """Selects budget samples uniformly at random from the unlabelled pool."""
    def __init__(self, seed=42):
        self.rng = np.random.default_rng(seed)

    def select(self, features, budget, already_labelled=None):
        N = len(features)
        mask = np.zeros(N, dtype=bool)
        if already_labelled is not None:
            mask[already_labelled] = True
        candidates = np.where(~mask)[0]
        return self.rng.choice(candidates, size=min(budget, len(candidates)), replace=False)

hist_random = run(RandomSelector(seed=SEED), 'Random')

## 6. Comparison Plot

In [ ]:
original = load_results('../results/tpcrp_original_history.json')

all_results = {
    'TPC_RP (original, proj=64)': original,
    'TPC_no_proj':                hist_no_proj,
    'Random':                     hist_random,
    **histories_proj,
}

plot_comparison(all_results, save_path='../plots/tpcrp_comparison.png')

## 7. Discussion

Fill in your analysis here after running the experiments:

- **Effect of random projection**: Does projecting to a lower dimension help or hurt selection quality?
- **Optimal projection dimension**: Which value of `proj_dim` performs best and why?
- **Comparison to random baseline**: At what label budget does TPC_RP start to outperform random sampling?
- **Limitations**: What are the failure modes of the typicality-based approach?